# 🚀 Qwen 27B OpenAI-Compatible Server — free Dual-T4 hosting (Kaggle)

Turn a **free Kaggle notebook session (2× NVIDIA T4, 32 GB VRAM)** into a public
**OpenAI-compatible LLM API** running a ~27B-parameter GGUF model with the native
C++ `llama-server`.

```
┌─────────────────────────────────────────────────────────────┐
│  Kaggle Notebook  (GPU T4 x2 — 32 GB VRAM total)            │
│                                                             │
│  llama-server (native C++, CUDA 12)                         │
│   ├─ ~27B GGUF — every layer offloaded to GPU (-ngl 99)     │
│   ├─ Flash Attention, optional quantized KV cache (q4_0)    │
│   ├─ Speculative decoding: draft model or embedded MTP      │
│   └─ OpenAI-compatible API on localhost:8001                │
│                    ▲                                        │
│  cloudflared quick tunnel ──►  https://<random>.trycloudflare.com
└─────────────────────────────────────────────────────────────┘
```

**What you get**

| Feature | Detail |
|---|---|
| OpenAI-compatible API | `/v1/chat/completions`, `/v1/completions`, `/v1/models` |
| Dual-GPU tensor parallelism | Both T4s work on every token (`-sm tensor`) |
| Speculative decoding | Auto: small draft GGUF if attached, else embedded MTP — 1.8–2.5× faster |
| 3 VRAM-tested profiles | ⚡ Speed 40K ctx · ⚖️ Balanced 64K · 🐘 MaxCtx 88K |
| Public HTTPS endpoint | Cloudflare quick tunnel, auto-reconnecting watchdog |
| Optional gateway mode | Register with your own router/worker for multi-node load balancing |

**Typical throughput on 2× T4** (27B, Q4-class quant): ~28–38 tok/s with embedded
MTP, ~50–65 tok/s with a small draft model attached.


## 📋 Before you run — checklist

1. **Accelerator** → Sidebar ▸ *Session options* ▸ **GPU T4 x2** (this notebook is tuned for exactly two T4s).
2. **Internet** → **On** (needed to download cloudflared and, on first run, to compile llama.cpp).
3. **Attach your model** → Sidebar ▸ *Input* ▸ *Upload/Create Dataset* containing one or more `.gguf` files:
   - **Largest `.gguf` found** becomes the main model (aim for a Q4-class quant ≈ 14–17 GB for 2× T4).
   - *(Optional)* a **second small GGUF** (100 MB – 6 GB, e.g. a 0.6B–0.8B model from the same family) becomes a **draft model** for speculative decoding.
   - *(Alternative)* set `KAGGLE_MODEL_HANDLE` in the config cell to pull a GGUF from Kaggle Models instead of attaching a dataset.
4. *(Optional)* **Attach a prebuilt `llama-server` binary** as a dataset to skip the ~15–20 min source build. The notebook auto-detects any `/kaggle/input/**/llama-server` file and saves the freshly-built binary to `/kaggle/working/llama_server_build/` so **you** can package it as a dataset next time.

> ⚠️ **Quota**: Kaggle gives ~30 GPU-hours/week. The watchdog below can shut the
> kernel down remotely if you wire it to your own gateway; otherwise just stop
> the session from the Kaggle UI when you're done.


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 1 · CONFIGURATION — the only cell you should need to edit
# ═════════════════════════════════════════════════════════════════════
# Everything has a sane default: attach a GGUF dataset and press "Run All".

MODEL_ALIAS   = "qwen-27b"   # model name your API clients will see
SERVER_PORT   = 8001         # llama-server's local port (behind the tunnel)

# ── Performance profile ─────────────────────────────────────────────
# KV cache type is the main lever on 2× T4 (32 GB):
#   · f16  KV = 256 KB/token — REQUIRED in tensor-split mode (fastest)
#   · q4_0 KV = 128 KB/token — needs a binary built with
#     GGML_CUDA_FA_ALL_QUANTS=ON (the source build below enables it)
#
#   profile     context   GPU split              KV     best for
#   ─────────   ───────   ────────────────────   ─────  ────────────────
#   speed        40K      tensor  (16,16)         f16    fastest tokens/s
#   balanced     64K      layer split (auto)     q4_0   best all-rounder
#   maxctx       88K      layer split (auto)     q4_0   long documents
PROFILE = "speed"             # "speed" | "balanced" | "maxctx"

# ── Speculative decoding ────────────────────────────────────────────
# "auto" → attach a second small GGUF (<6 GB) and it becomes a draft
#          model (~1.8–2.5× speedup). No draft attached? Use embedded
#          multi-token prediction (MTP) when the GGUF supports it.
#          Nothing applies? Plain decoding — always works.
# "off"  → disable speculative decoding entirely.
ENABLE_SPECULATIVE = "auto"

# ── Security ────────────────────────────────────────────────────────
# The tunnel URL is PUBLIC — anyone who obtains it can use your GPU
# session. Set a key to require `Authorization: Bearer <key>` on every
# request (llama-server enforces it). Leave "" only behind a gateway
# that already restricts access.
SERVER_API_KEY = ""

# ── Optional: pull the model from Kaggle Models instead of a dataset ─
# If no GGUF is attached, the notebook tries this handle via kagglehub.
# Example shape:  "owner/model-slug/gguf/quant-variant"  (your own or a
# public GGUF dataset). Leave "" to require an attached dataset.
KAGGLE_MODEL_HANDLE = ""

# ── Optional: register with YOUR central router/gateway ─────────────
# GATEWAY_MODE="register" turns this notebook into a worker node of
# your own router (it will register the tunnel URL and send heartbeats).
# Requires these Kaggle Secrets (Add-ons ▸ Secrets):
#   GATEWAY_URL    → https://your-worker.workers.dev
#   GATEWAY_TOKEN  → shared admin token
#   NODE_ID        → unique id, e.g. "node-1"
#   NODE_NAME      → display name, e.g. "My T4 Node"
# Standalone? Keep "none" and everything below still runs fine.
GATEWAY_MODE = "none"         # "none" (standalone) | "register"

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 2 · LLAMA-SERVER BINARY — attach a prebuilt one, or build from source
# ═════════════════════════════════════════════════════════════════════
# Fast path:   attach a dataset containing a CUDA `llama-server` binary
#              (+ its .so files). It's detected at /kaggle/input/**/llama-server.
# Slow path:   build natively from source (~15–20 min). The CMake flags
#              matter — GGML_CUDA_FA_ALL_QUANTS=ON is what allows the
#              quantized (q4_0) KV cache used by the balanced/maxctx profiles.
import os, re, sys, glob, time, json, shutil, subprocess, requests

print("🧹 Stopping leftover processes from earlier runs ...")
for _pat in ("llama-server", "llama_cpp", "cloudflared", "uvicorn"):
    subprocess.run(["pkill", "-9", "-f", _pat], check=False,
                   stderr=subprocess.DEVNULL)
time.sleep(1)

LLAMA_DIR        = "/kaggle/working/llama.cpp"
LLAMA_SERVER_BIN = f"{LLAMA_DIR}/build/bin/llama-server"

cached_bins = glob.glob("/kaggle/input/**/llama-server", recursive=True)
if cached_bins:
    # ── Reuse an attached binary: copy it + sibling .so libraries ──────
    src_bin = cached_bins[0]
    shutil.copy2(src_bin, "/kaggle/working/llama-server")
    LLAMA_SERVER_BIN = "/kaggle/working/llama-server"
    _dir = os.path.dirname(src_bin)
    for _f in os.listdir(_dir):
        if _f.endswith(".so") or ".so." in _f:
            try:
                shutil.copy2(os.path.join(_dir, _f), "/kaggle/working/" + _f)
            except Exception:
                pass
    os.chmod(LLAMA_SERVER_BIN, 0o755)
    print(f"⚡ Using attached binary from dataset: {os.path.basename(os.path.dirname(src_bin))}")
    print("⏩ Build skipped.\n")
else:
    print("🔧 Building llama-server from source (CUDA + FA for all quant types) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "ninja"], check=True)

    # Kaggle's driver stub lives in a nonstandard dir; symlink it where the
    # linker expects it so CMake's CUDA detection succeeds.
    _stubs = glob.glob("/usr/local/cuda*/**/libcuda.so*", recursive=True)
    if _stubs:
        os.makedirs("/usr/lib/x86_64-linux-gnu", exist_ok=True)
        subprocess.run(["ln", "-sf", _stubs[0], "/usr/lib/x86_64-linux-gnu/libcuda.so"],   check=False)
        subprocess.run(["ln", "-sf", _stubs[0], "/usr/lib/x86_64-linux-gnu/libcuda.so.1"], check=False)
    if os.path.exists(LLAMA_DIR):
        shutil.rmtree(LLAMA_DIR, ignore_errors=True)

    _t0 = time.time()
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggerganov/llama.cpp", LLAMA_DIR], check=True)
    subprocess.run(["cmake", "-B", f"{LLAMA_DIR}/build", "-S", LLAMA_DIR, "-G", "Ninja",
        "-DGGML_CUDA=ON",                    # CUDA backend — required for GPU offload
        "-DGGML_CUDA_FA_ALL_QUANTS=ON",      # flash-attention for QUANTIZED KV (q4_0) too
        "-DCMAKE_BUILD_TYPE=Release",
        "-DLLAMA_BUILD_SERVER_UI=OFF",       # trim what we don't need → faster build
        "-DLLAMA_BUILD_TESTS=OFF",
        "-DLLAMA_BUILD_EXAMPLES=OFF"], check=True)

    # Build with live progress (full output → log file, we print % only)
    BUILD_LOG = "/tmp/llama_build.log"
    _jobs = os.cpu_count() or 4
    _proc = subprocess.Popen(
        ["cmake", "--build", f"{LLAMA_DIR}/build", "--target", "llama-server",
         "-j", str(_jobs)],
        stdout=open(BUILD_LOG, "w"), stderr=subprocess.STDOUT)
    _last = 0
    while _proc.poll() is None:
        time.sleep(5)
        try:
            with open(BUILD_LOG) as f:
                _m = re.findall(r"\[(\d+)/(\d+)\]", f.read())
            if _m:
                _done, _tot = int(_m[-1][0]), int(_m[-1][1])
                _pct = _done * 100 // _tot
                if _pct != _last:
                    print(f"   🔨 Build {_m[-1][0]}/{_m[-1][1]} ({_pct}%)", flush=True)
                    _last = _pct
        except Exception:
            pass
    _proc.wait()
    if _proc.returncode != 0:
        with open(BUILD_LOG) as f:
            print("❌ Build failed. Last log lines:\n" + f.read()[-2000:])
        sys.exit(1)

    # Locate the freshly built binary (path moved across llama.cpp versions)
    for _p in [f"{LLAMA_DIR}/build/bin/llama-server",
               f"{LLAMA_DIR}/build/src/llama-server",
               f"{LLAMA_DIR}/build/**/llama-server"]:
        _hits = glob.glob(_p)
        if _hits and os.path.exists(_hits[0]):
            LLAMA_SERVER_BIN = _hits[0]
            os.chmod(LLAMA_SERVER_BIN, 0o755)
            break
    if not LLAMA_SERVER_BIN or not os.path.exists(LLAMA_SERVER_BIN):
        print("❌ Could not find llama-server binary after build!")
        sys.exit(1)

    print(f"✅ Built in {time.time()-_t0:.0f}s  "
          f"({os.path.getsize(LLAMA_SERVER_BIN)/(1024**2):.0f} MB)\n")

    # ── Save the binary to /kaggle/working so YOU can package it as a ──
    #    dataset and skip this build on every future run.
    try:
        _exp = "/kaggle/working/llama_server_build"
        os.makedirs(_exp, exist_ok=True)
        shutil.copy(LLAMA_SERVER_BIN, f"{_exp}/llama-server")
        os.chmod(f"{_exp}/llama-server", 0o755)
        for _so in (glob.glob(f"{LLAMA_DIR}/build/bin/**/*.so*", recursive=True) +
                    glob.glob(f"{LLAMA_DIR}/build/**/*.so*", recursive=True)):
            if os.path.isfile(_so):
                try: shutil.copy(_so, _exp)
                except Exception: pass
        print(f"📦 Binary staged in /kaggle/working/llama_server_build/ "
              f"→ create a Dataset from it after the run to skip future builds.\n")
    except Exception as e:
        print(f"⚠️ Could not stage binary for reuse: {e}\n")

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 3 · FIND THE MODEL — largest GGUF = target, second small one = drafter
# ═════════════════════════════════════════════════════════════════════
print("🔍 Scanning /kaggle/input/ for GGUF models ...")

def find_all_ggufs():
    found = []
    if os.path.exists("/kaggle/input"):
        for root, _dirs, files in os.walk("/kaggle/input"):
            for fname in files:
                fpath = os.path.join(root, fname)
                try:
                    fsize = os.path.getsize(fpath)
                    # Real weights only: >50 MB, no projector / importance matrix
                    if fsize > 50 * (1024**2) and "mmproj" not in fname.lower() \
                       and "imatrix" not in fname.lower() and fname.lower().endswith(".gguf"):
                        found.append(fpath)
                except Exception:
                    pass
    return found

all_ggufs = find_all_ggufs()

# Fallback: no dataset attached → try kagglehub with the configured handle
if not all_ggufs and KAGGLE_MODEL_HANDLE:
    print(f"⬇️  No GGUF in /kaggle/input — downloading via kagglehub: {KAGGLE_MODEL_HANDLE}")
    try:
        import kagglehub
        _dir = kagglehub.model_download(KAGGLE_MODEL_HANDLE)
        for _f in os.listdir(_dir):
            if _f.endswith(".gguf"):
                all_ggufs.append(os.path.join(_dir, _f))
    except Exception as e:
        print(f"⚠️ kagglehub download failed: {e}")

if not all_ggufs:
    print("❌ No GGUF model found! Attach a model dataset (or set KAGGLE_MODEL_HANDLE).")
    sys.exit(1)

# ── Main model = biggest file (>8 GB expected for a 27B Q4) ──────────
_target = [f for f in all_ggufs if os.path.getsize(f) > 8 * (1024**3)]
if not _target:
    _target = [sorted(all_ggufs, key=os.path.getsize, reverse=True)[0]]
MODEL_PATH = _target[0]
MODEL_GB   = os.path.getsize(MODEL_PATH) / (1024**3)

MODEL_QUANT = next((q for q in ("UD-Q4_K_XL", "Q4_K_M", "Q5_K_M", "Q6_K")
                    if q in MODEL_PATH), "Q4")
print(f"✅ Main model [{MODEL_QUANT}]: {os.path.basename(MODEL_PATH)} ({MODEL_GB:.2f} GB)")

# ── Draft model = a second, much smaller GGUF (100 MB … 6 GB) ────────
DRAFT_MODEL_PATH = None
_drafts = [f for f in all_ggufs
           if f != MODEL_PATH and 100 * 1024**2 <= os.path.getsize(f) <= 6 * (1024**3)]
if ENABLE_SPECULATIVE == "auto" and _drafts:
    DRAFT_MODEL_PATH = _drafts[0]
    print(f"🚀 Speculative decoding: external draft model "
          f"({os.path.basename(DRAFT_MODEL_PATH)}, "
          f"{os.path.getsize(DRAFT_MODEL_PATH)/(1024**2):.0f} MB) → expect ~50-65 tok/s")
elif ENABLE_SPECULATIVE == "auto":
    print("🚀 Speculative decoding: embedded MTP if the GGUF has MTP heads "
          "(n=3, p=0.1) → expect ~28-38 tok/s  (attach a small GGUF for more speed)")
else:
    print("ℹ️ Speculative decoding disabled (ENABLE_SPECULATIVE='off')")

# Rule-of-thumb VRAM ledger for 2× T4 (32 GB):
#   weights(Q4 ≈ model GB) + KV(256 KB/token @f16 · 128 KB/token @q4_0)
print(f"   Weights {MODEL_GB:.1f} GB + KV cache → see profile table in cell 1.")
print()

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 4 · LAUNCH LLAMA-SERVER  (the heart of the notebook)
# ═════════════════════════════════════════════════════════════════════
# llama-server = llama.cpp's OpenAI-compatible HTTP server. Key flags:
#   -ngl 99        offload ALL layers to GPU
#   -sm tensor     tensor-parallel split: BOTH T4s compute every token
#                  (only works with f16 KV → that's the "speed" profile)
#   no -sm flag    layer split (default): layers alternate across GPUs,
#                  accepts q4_0 KV → half the KV memory → bigger context
#   -fa on         flash attention (required for quantized KV)
#   -c N           total KV context (tokens)
#   --jinja        use the model's chat template as-is (tool calls etc.)

PROFILES = {
    #        total ctx   split strategy                      KV cache   label
    "speed":    {"ctx": 40960, "split": ["-sm", "tensor", "-ts", "16,16"],
                 "kv": ["-ctk", "f16",  "-ctv", "f16"],  "label": "⚡ SPEED 40K"},
    "balanced": {"ctx": 65536, "split": [],
                 "kv": ["-ctk", "q4_0", "-ctv", "q4_0"], "label": "⚖️ BALANCED 64K"},
    "maxctx":   {"ctx": 90112, "split": [],
                 "kv": ["-ctk", "q4_0", "-ctv", "q4_0"], "label": "🐘 MAXCTX 88K"},
}
ACTIVE_PROFILE = PROFILE if PROFILE in PROFILES else "speed"
if PROFILE not in PROFILES:
    print(f"⚠️ Unknown profile '{PROFILE}' — falling back to 'speed'")
P = PROFILES[ACTIVE_PROFILE]
PARALLEL_SLOTS = 1

def build_server_cmd(profile_name, use_spec=True):
    """Assemble the llama-server command line for a profile."""
    p = PROFILES[profile_name]

    spec_flags = []
    if use_spec and ENABLE_SPECULATIVE == "auto":
        if DRAFT_MODEL_PATH:
            spec_flags = [                       # classic speculative decoding
                "-md", DRAFT_MODEL_PATH,         # draft model file
                "-ngld", "99",                   # draft fully on GPU
                "--spec-type", "draft",
                "--spec-draft-n-max", "4",       # up to 4 drafted tokens
                "--spec-draft-p-min", "0.15"]    # min acceptance prob
        else:
            spec_flags = [                       # multi-token prediction, if in GGUF
                "--spec-type", "draft-mtp",
                "--spec-draft-n-max", "3",
                "--spec-draft-p-min", "0.1"]

    return [
        LLAMA_SERVER_BIN, "-m", MODEL_PATH, "-a", MODEL_ALIAS,
        "--host", "0.0.0.0", "--port", str(SERVER_PORT),
        "-ngl", "99",                            # all model layers → GPU
        *p["split"],
        *(["--fit", "off"] if p["split"] else []),  # auto-fit unsupported in tensor mode
        "-c", str(p["ctx"] * PARALLEL_SLOTS),    # total KV cache tokens
        "-fa", "on",                             # flash attention
        *p["kv"],                                # KV cache quant types
        "-b", "2048", "-ub", "512",              # logical/physical batch sizes
        "-np", str(PARALLEL_SLOTS),
        "--cont-batching", "--cache-prompt",     # continuous batching + prompt cache
        "--mlock", "--no-mmap",                  # pin weights in RAM
        "--metrics", "--jinja",
        *spec_flags,
    ]

LOG_FILE = "/tmp/llama_server.log"

def wait_for_health(proc, timeout=300):
    """Stream interesting log lines while polling the /health endpoint."""
    _t0, _pos = time.time(), 0
    while time.time() - _t0 < timeout:
        if os.path.exists(LOG_FILE):
            with open(LOG_FILE) as f:
                f.seek(_pos); new = f.read()
            if new:
                for line in new.splitlines():
                    if any(k in line.lower() for k in
                           ("listening", "http://", "error", "failed", "cuda", "mtp", "draft")):
                        print(f"   {line.strip()}")
                _pos += len(new)
        if proc.poll() is not None:
            return False                          # process died
        try:
            if requests.get(f"http://127.0.0.1:{SERVER_PORT}/health",
                            timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False

server_env = os.environ.copy()
server_env["GGML_CUDA_PEER_MAX_BATCH_SIZE"] = "512"   # NVLink-less P2P tuning
_ld = server_env.get("LD_LIBRARY_PATH", "")
server_env["LD_LIBRARY_PATH"] = "/kaggle/working:" + _ld if _ld else "/kaggle/working"

print(f"🚀 Starting llama-server — {P['label']} | Slots: {PARALLEL_SLOTS}")
print("⏳ Loading model into VRAM ...")

log_fh        = open(LOG_FILE, "w")
use_spec      = ENABLE_SPECULATIVE == "auto"
server_cmd    = build_server_cmd(ACTIVE_PROFILE, use_spec)
server_proc   = subprocess.Popen(server_cmd, stdout=log_fh,
                                 stderr=subprocess.STDOUT, env=server_env)

_t0       = time.time()
launch_ok = wait_for_health(server_proc)

# One retry — and if the binary rejected a speculative flag, relaunch without it
if not launch_ok and server_proc.poll() is not None:
    with open(LOG_FILE) as f:
        _tail = f.read()[-3000:]
    if "unknown option" in _tail or "invalid argument" in _tail or "unrecognized" in _tail:
        print("⚠️ This build rejects the speculative-decoding flags — retrying without them.")
        use_spec   = False
        server_cmd = build_server_cmd(ACTIVE_PROFILE, use_spec=False)
        log_fh     = open(LOG_FILE, "w")
        server_proc = subprocess.Popen(server_cmd, stdout=log_fh,
                                       stderr=subprocess.STDOUT, env=server_env)
        launch_ok = wait_for_health(server_proc)

if not launch_ok:
    print("❌ Server failed to start. Last log lines:")
    try:
        with open(LOG_FILE) as f:
            print(f.read()[-2000:])
    except Exception:
        pass
    server_proc.terminate(); sys.exit(1)

print(f"✅ Model loaded & healthy in {time.time()-_t0:.0f}s!")
try:
    _m = requests.get(f"http://127.0.0.1:{SERVER_PORT}/v1/models", timeout=5).json()
    print(f"   Models: {[m['id'] for m in _m.get('data', [])]}")
except Exception:
    pass

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 5 · PUBLIC HTTPS TUNNEL (Cloudflare quick tunnel — no account needed)
# ═════════════════════════════════════════════════════════════════════
# Kaggle has no inbound networking, so we run `cloudflared` in "quick
# tunnel" mode: it gives us a random https://<something>.trycloudflare.com
# URL that forwards to localhost:8001. The URL changes every session.

CLOUDFLARED_BIN = "/tmp/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    print("🌐 Downloading cloudflared ...")
    subprocess.run(["curl", "-sL", "-o", CLOUDFLARED_BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True)
    os.chmod(CLOUDFLARED_BIN, 0o755)

def start_tunnel():
    return subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def get_tunnel_url(proc, timeout=30):
    """Read cloudflared's output until the trycloudflare URL appears."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if line:
            if "api.trycloudflare.com" in line or "pkg.cloudflare.com" in line:
                continue
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m and "api.trycloudflare.com" not in m.group(0):
                return m.group(0)
        else:
            time.sleep(0.3)
    return None

tunnel_proc = start_tunnel()
tunnel_url  = get_tunnel_url(tunnel_proc)
if not tunnel_url:
    print("❌ Could not obtain a tunnel URL — re-run this cell.")
    tunnel_proc.terminate(); sys.exit(1)

print(f"🌐 Tunnel: {tunnel_url}")

# Verify the tunnel actually forwards to the server
tunnel_ok = False
for _ in range(10):
    try:
        if requests.get(f"{tunnel_url}/health", timeout=5).status_code == 200:
            print("✅ Tunnel verified end-to-end!")
            tunnel_ok = True
            break
    except Exception:
        time.sleep(1)
if not tunnel_ok:
    print("⚠️ Tunnel /health not answering yet — it usually warms up in a few seconds.")

print()
print("=" * 66)
print(f"   YOUR PUBLIC API URL:  {tunnel_url}")
print(f"   Model name:           {MODEL_ALIAS}")
print("=" * 66)

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 6 · SMOKE TEST — prove the API works end-to-end (via the tunnel)
# ═════════════════════════════════════════════════════════════════════
_hdrs = {"Authorization": f"Bearer {SERVER_API_KEY}"} if SERVER_API_KEY else {}
try:
    _r = requests.post(f"{tunnel_url}/v1/chat/completions", headers=_hdrs, timeout=180, json={
        "model": MODEL_ALIAS,
        "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
        "max_tokens": 64, "temperature": 0.7})
    _r.raise_for_status()
    print("✅ API is LIVE — sample reply:")
    print("   " + _r.json()["choices"][0]["message"]["content"].strip())
except Exception as e:
    print(f"❌ Smoke test failed: {e}")
    print("   → check the server log:  !tail -40 /tmp/llama_server.log")

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# 7 · KEEP-ALIVE + optional GATEWAY MODE
# ═════════════════════════════════════════════════════════════════════
# This cell runs forever (that's what keeps your session alive).
#   · GATEWAY_MODE="none"      → standalone: prints a metrics heartbeat.
#   · GATEWAY_MODE="register"  → also registers with YOUR gateway and
#                                answers its heartbeats (remote stop /
#                                profile switch), using the secrets
#                                GATEWAY_URL / GATEWAY_TOKEN / NODE_ID / NODE_NAME.
# Stop everything with the notebook's interrupt (■) button.

def get_metrics():
    """Pull tokens/sec from llama-server's Prometheus /metrics endpoint."""
    try:
        t = requests.get(f"http://127.0.0.1:{SERVER_PORT}/metrics", timeout=3).text
        pp = re.search(r"prompt_tokens_per_second\{[^}]*\}\s+([\d.]+)", t) \
             or re.search(r"prompt_tokens_per_second\s+([\d.]+)", t)
        tg = re.search(r"predicted_tokens_per_second\{[^}]*\}\s+([\d.]+)", t) \
             or re.search(r"predicted_tokens_per_second\s+([\d.]+)", t)
        return (float(pp.group(1)) if pp else 0.0,
                float(tg.group(1)) if tg else 0.0)
    except Exception:
        return 0.0, 0.0

def shutdown_kernel():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip:
            ip.kernel.do_shutdown(restart=False)   # stops the GPU-quota timer
    except Exception:
        pass

# ---------- optional: register with your own gateway ----------
gateway_url = gateway_token = node_id = node_name = None
if GATEWAY_MODE == "register":
    try:
        from kaggle_secrets import UserSecretsClient
        _s   = UserSecretsClient()
        gateway_url   = _s.get_secret("GATEWAY_URL").rstrip("/")
        gateway_token = _s.get_secret("GATEWAY_TOKEN")
        node_id       = _s.get_secret("NODE_ID")
        node_name     = _s.get_secret("NODE_NAME") or node_id
        _r = requests.post(f"{gateway_url}/admin/register-backend",
            headers={"Authorization": f"Bearer {gateway_token}"},
            json={"backend_url": tunnel_url, "account_id": node_id,
                  "account_name": node_name, "model": MODEL_ALIAS,
                  "gpu_type": f"Dual-T4 | {P['label']} | {MODEL_QUANT}"},
            timeout=15)
        if _r.status_code == 200:
            print(f"🚀 Registered with gateway — node '{node_name}' is LIVE")
        else:
            print(f"⚠️ Gateway registration: {_r.status_code} — {_r.text[:200]}")
    except Exception as e:
        print(f"⚠️ Gateway registration failed: {e}")

# ---------- watchdog loop ----------
gpu_type_str = (f"Dual-T4 | {P['label']} | "
                f"{os.path.basename(DRAFT_MODEL_PATH) if DRAFT_MODEL_PATH else ('MTP' if use_spec else 'no-spec')} | "
                f"{MODEL_QUANT} | {PARALLEL_SLOTS}-slot")
print(f"\n{'='*66}\n   SERVING — interrupt (■) this cell to shut down\n   {P['label']} | speculative: {'draft' if DRAFT_MODEL_PATH else ('MTP' if use_spec else 'off')} | slots: {PARALLEL_SLOTS}\n{'='*66}\n")

hb_count, crash_restarts = 0, 0
try:
    while True:
        time.sleep(30)
        hb_count += 1

        # self-heal: restart the server if it crashed
        if server_proc.poll() is not None:
            print(f"\n❌ llama-server exited (code {server_proc.returncode}) — restarting ...")
            if crash_restarts >= 5:
                print("💀 Too many crashes — giving up.")
                break
            crash_restarts += 1
            time.sleep(2)
            log_fh      = open(LOG_FILE, "w")
            server_cmd  = build_server_cmd(ACTIVE_PROFILE, use_spec)
            server_proc = subprocess.Popen(server_cmd, stdout=log_fh,
                                           stderr=subprocess.STDOUT, env=server_env)
            if wait_for_health(server_proc):
                print("✅ Recovered & healthy.")

        # self-heal: restart the tunnel if cloudflared died
        if tunnel_proc.poll() is not None:
            print("\n❌ Tunnel died — restarting ...")
            tunnel_proc = start_tunnel()
            _new = get_tunnel_url(tunnel_proc, timeout=20)
            if _new:
                tunnel_url = _new
                print(f"   New tunnel: {tunnel_url}")

        pp, tg = get_metrics()

        # optional gateway heartbeat + remote commands
        if gateway_url:
            try:
                _h = requests.post(f"{gateway_url}/admin/heartbeat",
                    headers={"Authorization": f"Bearer {gateway_token}"},
                    json={"account_id": node_id, "status": "healthy",
                          "pp_tps": round(pp, 1), "tg_tps": round(tg, 1)}, timeout=10)
                if _h.status_code == 200:
                    _d = _h.json()
                    if _d.get("command") in ("force_kill", "shutdown") \
                       or _d.get("status") == "stopped":
                        print("\n🛑 Remote stop received — shutting down.")
                        server_proc.terminate(); tunnel_proc.terminate()
                        shutdown_kernel(); os._exit(0)
                    if _d.get("command") == "reconfigure":
                        _want = int(_d.get("context_size", P["ctx"]))
                        _tgt  = min(PROFILES, key=lambda n: abs(PROFILES[n]["ctx"] - _want))
                        if _tgt != ACTIVE_PROFILE:
                            print(f"🔄 Reconfigure: {P['label']} → {PROFILES[_tgt]['label']}")
                            ACTIVE_PROFILE = _tgt; P = PROFILES[_tgt]
                            server_proc.terminate()
                            try: server_proc.wait(timeout=8)
                            except Exception: pass
                            server_cmd  = build_server_cmd(_tgt, use_spec)
                            server_proc = subprocess.Popen(server_cmd, stdout=log_fh,
                                                           stderr=subprocess.STDOUT, env=server_env)
                            if wait_for_health(server_proc):
                                print(f"✅ Now serving {P['label']} ({P['ctx']//1024}K ctx)")
            except Exception as e:
                print(f"⚠️ Heartbeat: {e}")

        # local heartbeat line every ~5 minutes
        if hb_count % 10 == 0:
            print(f"💓 #{hb_count:>4} | prefill {pp:6.1f} tok/s | gen {tg:6.1f} tok/s")

except KeyboardInterrupt:
    print("\n🛑 Shutting down ...")
finally:
    try: server_proc.terminate()
    except Exception: pass
    try: tunnel_proc.terminate()
    except Exception: pass
    print("✅ Server and tunnel stopped.")

## 🧪 Using your server

```bash
curl https://<your-tunnel>.trycloudflare.com/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer <SERVER_API_KEY if set>" \
  -d '{
    "model": "qwen-27b",
    "messages": [{"role": "user", "content": "Explain speculative decoding briefly."}]
  }'
```

Or with any OpenAI SDK:

```python
from openai import OpenAI
client = OpenAI(base_url="https://<your-tunnel>.trycloudflare.com/v1",
                api_key="<SERVER_API_KEY or 'none'>")
resp = client.chat.completions.create(
    model="qwen-27b",
    messages=[{"role": "user", "content": "Hello!"}])
print(resp.choices[0].message.content)
```

## 🔧 Troubleshooting

| Symptom | Fix |
|---|---|
| CUDA out of memory during load | Switch `PROFILE` to `"speed"` (smaller KV), or attach a smaller quant |
| Build fails / takes forever | Attach a prebuilt `llama-server` binary dataset (detected automatically) |
| Server exits with unknown-option error | Handled automatically — the notebook relaunches without speculative flags |
| Tunnel URL dead after a while | The watchdog restarts it; the **URL changes**, update your client |
| Very slow generation | Attach a small draft GGUF (same model family, 0.6–0.8B) for ~2× speedup |
| `--mlock` warning in log | Harmless on Kaggle — it means RAM pinning was capped |

## 📝 Notes & honest caveats

- **The quick-tunnel URL is public.** Anyone who learns it can use your session — set `SERVER_API_KEY` in the config cell for standalone use.
- **Context limits are VRAM physics**: tensor mode requires f16 KV (256 KB/token → ~40K on 2× T4); quantized q4_0 KV (128 KB/token) reaches 64K–88K but needs a binary built with `GGML_CUDA_FA_ALL_QUANTS=ON` — exactly what the source build in step 2 produces.
- **Embedded MTP** only activates when your GGUF actually embeds multi-token-prediction heads; otherwise the notebook silently falls back to plain (or draft-model) decoding. No flags are forced.
- **Ephemeral by design**: sessions stop when idle (~40–60 min without browser interaction), the tunnel URL rotates, and `/kaggle/working` is wiped — persist your binary as a Kaggle Dataset for instant restarts.
- **GPU quota**: ~30 h/week on the free tier. Interrupt the keep-alive cell (or stop the session) when you're done.
